## 02 - Chunk unified samples
Load unified QA samples, apply a chosen chunker, and save chunk records for later retrieval evaluation.

In [1]:
# Robustly set repo root and import path
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for cand in [ROOT, *ROOT.parents]:
    if (cand / "src").exists():
        ROOT = cand
        break
else:
    raise RuntimeError("Could not locate repo root containing src/.")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print(f"Repo root: {ROOT}")


Repo root: C:\uni\chunks\Chunking-Research


### Set paths and imports
Ensures the notebook can import the repository modules from `src`.

In [2]:
# ---- User choices ----
samples_path = ROOT / 'data/processed/demo_samples.jsonl'
chunker_name = 'fixed_size'  # fixed_size | passage | text_tiling
chunker_params = { 'chunk_size': 300, 'overlap': 40 }  # override defaults as needed

chunks_output_path = ROOT / 'results/demo_chunks_precomputed.jsonl'
overwrite = False

### Configure chunking run
Point to the unified samples file, pick a chunker and params, and set where to save chunk records.

In [3]:
from src.data_loader import load_samples_jsonl

samples = load_samples_jsonl(str(samples_path))
print(f'Loaded {len(samples)} samples from {samples_path}')
print('First sample question:', samples[0].question)


Loaded 1000 samples from C:\uni\chunks\Chunking-Research\data\processed\demo_samples.jsonl
First sample question: Co było powodem powrócenia konceptu porozumieniu monachijskiego?


### Load unified samples
Read the JSONL produced in Notebook 01 and sanity-check the first question.

In [4]:
from src.chunking import get_chunker
from src.schemas import ChunkRecord

chunker = get_chunker(chunker_name, chunker_params)
chunk_records = []
total_chunks = 0

for sample in samples:
    chunks = chunker.split_text(sample.context, document_meta={ 'sample_id': sample.sample_id })
    total_chunks += len(chunks)
    for ch in chunks:
        chunk_records.append(ChunkRecord.from_chunk(ch, sample.sample_id))

print(f'Produced {total_chunks} chunks across {len(samples)} samples')
print('First chunk record:', chunk_records[0].to_dict() if chunk_records else {})


Produced 3793 chunks across 1000 samples
First chunk record: {'chunk_id': '2820b544-c201-4e88-98ed-faf2b986cbbe', 'text': 'Projekty konfederacji zaczęły się załamywać 5 sierpnia 1942. Ponownie wróciła kwestia monachijska, co uaktywniło się wymianą listów Ripka – Stroński. Natomiast 17 sierpnia 1942 doszło do spotkania E. Beneša i J. Masaryka z jednej a Wł. Sikorskiego i E. Raczyńskiego z drugiej strony. Polscy dyplomaci', 'metadata': {'sample_id': '1', 'start_char': 0, 'end_char': 300}}


### Chunk each context
Initialize the chosen chunker, generate chunk records with sample_ids, and inspect the first one.

In [5]:
from run_chunking import ensure_output_path
from src.schemas import save_chunk_records_jsonl

path_to_write = ensure_output_path(str(chunks_output_path), overwrite)
save_chunk_records_jsonl(chunk_records, str(path_to_write))
print(f'Saved chunks to {path_to_write}')


Saved chunks to C:\uni\chunks\Chunking-Research\results\demo_chunks_precomputed.jsonl


### Save chunk records for reuse
Writes chunk JSONL that can be plugged into retrieval evaluation without re-chunking.